## 1、ChatTemplate的介绍

## 1.1、训练的时候，调用chat template

In [4]:
from transformers import AutoTokenizer
from llm_config import ModelConfig
org_model_name = ModelConfig.REMOTE_MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(org_model_name)
message_list = [
    {"role":"user","content":"你好"},
     {"role":"assistant","content":"你也好"},
]
res = tokenizer.apply_chat_template(message_list,tokenize=True)
print(res)

{'input_ids': [151644, 872, 198, 108386, 151645, 198, 151644, 77091, 198, 151667, 271, 151668, 271, 56568, 106920, 151645, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [3]:
from dataclasses import dataclass

@dataclass
class SFTConfig:
    train_data_size: int  = 10000
    eval_data_size: int = 500
    lr: float = 3e-5
    batch_size: int = 4
    warmup_ratio: float = 0.1
    save_dir: str = "./finetuned/02_sft_demo"
    log_dir :str = "./logs/02_sft_demo"

    eval_iter:int = 100

    log_iter: int = 100




In [6]:
def get_train_data(sft_config:SFTConfig):

    from datasets import load_dataset
    train_data = load_dataset("./data/ultrachat_200k")["train_sft"]
    train_data = train_data.shuffle()
    train_data = train_data.select(range(sft_config.train_data_size))
    final_result = []
    for i in range(sft_config.train_data_size):
        message_list = train_data[i]["messages"]
        result:list = tokenizer.apply_chat_template(message_list,tokenize=True)["input_ids"]
        final_result.append(result)

    return final_result

def get_eval_data(sft_config:SFTConfig):

    from datasets import load_dataset
    eval_data = load_dataset("./data/ultrachat_200k")["test_sft"]
    eval_data = eval_data.shuffle()
    eval_data = eval_data.select(range(sft_config.eval_data_size))
    final_result = []
    for i in range(sft_config.eval_data_size):
        message_list = eval_data[i]["messages"]
        result:list = tokenizer.apply_chat_template(message_list,tokenize=True)["input_ids"]
        final_result.append(result)

    return final_result

## 1.2、推理阶段

In [7]:
message_list = [
    {"role":"user","content":"你好呀！"}
]
res = tokenizer.apply_chat_template(message_list,tokenize=False,add_generation_prompt=True)
print(res)

<|im_start|>user
你好呀！<|im_end|>
<|im_start|>assistant



# 2、构造assistant_answer_mask

In [ ]:
from transformers import PreTrainedTokenizerFast
from typing import List
import torch
def create_answer_mask(labels,tokenizer:PreTrainedTokenizerFast):
    """
    创建answer mask，从labels当中找出assistant回答的部分，然后输出一个与labels相同shape的mask
    """
    # 构建answer mask，输入的labels为批量 tokenize之后的数据，对于每一条数据，查找当中assistant回答的部分，将其设置为1

    # 1. 构造一个和labels相同shape的全0矩阵
    answer_mask = torch.zeros_like(labels)

    # 2、找到<|im_end|> 所对应的token_id
    eos_token_id = tokenizer.encode("<|im_end|>")[0]

    # 3、遍历labels中的每一个样本
    # labels.shape: batch_size, seq_len
    for idx,ids in enumerate(labels):
        # 3.1、获取到所有的eos_position
        eos_position:List = torch.where(ids == eos_token_id)[0].tolist()
        # 3.2、解析获得user_ends和assistant_ends
        user_ends,assistant_ends = _parse_conversation_turns(eos_position)
        # 3.3、设置answer mask
        _set_answer_masks(answer_mask[idx],user_ends,assistant_ends)   
    
    # 4、结果返回:
    return answer_mask

def _parse_conversation_turns(eos_positions:List[int]):
    """
    输入eos_positions，输出user所对应的end位置和assistant所对应的end位置。

    以下面的对话为例：
    <|im_start|>user
    什么是习惯？<|im_end|>
    <|im_start|>assistant
    习惯是指在一定时间内重复执行的行为。<|im_end|>
    <|im_start|>user
    如何培养一个习惯<|im_end|>
    <|im_start|>assistant
    21天培养法，每天坚持xxx<|im_end|>

    假设第一个eos_token_id index为10，第二个为15，第三个为20，第四个为25
    那么输入的eos_token_id为：[10,15,20,25]
    user_turns为从第一个开始取，每隔一个取一次，assistant_turns为从第二个开始取，每隔一个取一次。

    输出结果为：
        user_turns:[10,20]
        assistant_ends:[15,25]
    """

    use_ends = [pos for pos in eos_positions[::2]]
    assistant_ends = [pos for pos in eos_positions[1::2]]

    return use_ends,assistant_ends

def _set_answer_masks(mask,user_ends,assistant_ends):
    """
    将mask当中，assistant回答的部分，设置为1（原地修改，不返回新的mask），其余部分保持为0

    以下面的对话为例：
    <|im_start|>user
    什么是习惯？<|im_end|>
    <|im_start|>assistant
    习惯是指在一定时间内重复执行的行为。<|im_end|>
    <|im_start|>user
    如何培养一个习惯<|im_end|>
    <|im_start|>assistant
    21天培养法，每天坚持xxx<|im_end|>

    假设第一个eos_token_id index为10，第二个为15，第三个为20，第四个为25
    那么user_turns:[10,20]，assistant_ends:[15,25]

    
    要想获取到assistant的回答的起始位置，就需要跳过<|im_end|>, \n, <|im_start|>,assistant , \n 这5个token
    要想获取到assistant的回答的结束位置，需要将<|im_end|>也包括进去，又因为列表切片是左闭右开的，所以需要向后移动一位
    """
    num_user_turns = len(user_ends)
    num_assistant_turns = len(assistant_ends)
    # 多轮对话没有被截断或者最后一轮整个assistant回答被截断，user轮数和assistant轮数一致
    if num_user_turns == num_assistant_turns:
        for user_end,assistant_end in zip(user_ends,assistant_ends):
            answer_start = user_end + 5
            answer_end = assistant_end + 1
            mask[answer_start:answer_end] = 1

    # 最后一轮，assistant回答被部分截断，此时user轮数比assistant轮数多一轮
    elif num_user_turns == num_assistant_turns + 1:
        for user_end,assistant_end in zip(user_ends[:-1],assistant_ends):
            answer_start = user_end + 5
            answer_end = assistant_end + 1
            mask[answer_start:answer_end] = 1
        
        # 处理最后一轮被截断的助手回答
        last_user_end = user_ends[-1] 
        last_answer_start = last_user_end + 5
        mask[last_answer_start:] = 1


In [ ]:
message_list = [
    {"role":"user","content":"你好"},
     {"role":"assistant","content":"你也好"},
     {"role":"user","content":"早上好"},
     {"role":"assistant","content":"你也早上好"},
]
res = tokenizer.apply_chat_template(message_list, tokenize=True)

In [ ]:
input_ids = torch.tensor([res["input_ids"]])
input_ids

In [ ]:
labels = input_ids[:,1:]
assistant_answer_mask = create_answer_mask(labels=labels,tokenizer=tokenizer)

In [ ]:
# 验证 assistant_answer_mask为1的token，是否真的assistant的回答：
for label_id, assistant_mask in zip(labels[0],assistant_answer_mask[0]):
    print("当前的label_id是，",repr(tokenizer.decode(label_id)),"，当前的label是否为assistant answer：",assistant_mask)

# 3、SFT的损失计算函数

In [ ]:
def compute_loss(logits,labels,assistant_mask):
    """
    SFT的损失计算函数,
    logits: 模型前向传播所得的结果，logits.shape: [batch_size, num_tokens, vocab_size]
    labels: 答案，shape：[batch_size, num_tokens]
    assistant_mask: 模型回答掩码, shape: [batch_size, num_tokens]
    """
    # 1、获取到log_probs，shape:  [batch_size, num_tokens, vocab_size]
    log_probs = torch.log_softmax(logits, dim=-1)

    # 2、需要从log_probs里面找到，输出答案token的具体的对数概率
    # 此处需要使用：torch.gather算子
    # shape: batch_size, num_tokens
    label_token_log_prob = torch.gather(
        input=log_probs,
        dim=-1,
        index=labels.unsqueeze(-1)
    ).squeeze(-1)

    # 3、取负值，获取负对数概率
    negative_label_token_log_prob = label_token_log_prob * (-1)

    # 4、对负对数概率，进行掩码，让非 assistant answer 部分，置为0，assistant answer，保留原值
    masked_negative_label_token_log_prob  = assistant_mask * negative_label_token_log_prob

    # 5、对负对数概率相加，取平均
    # shape: 普通的标量
    average_loss = masked_negative_label_token_log_prob.sum() / assistant_mask.sum()

    return average_loss


## 3.1 torch.gather的介绍

In [ ]:
log_probs = torch.tensor([[[0.1, 0.2 , 0.7],[0.3, 0.5, 0.2],[0.9,0.05, 0.05]]])
labels = torch.tensor([[0, 2, 1]])
res = torch.gather(
    input=log_probs, 
    dim=-1,
    index=labels.unsqueeze(-1)
    )
res.squeeze(-1)

# 4、训练主流程

## 4.1 学习率衰减的函数

In [ ]:
# cosine_decay(current_batch,total_batch,sft_config.warmup_ratio,sft_config.lr)
import numpy as np
def cosine_decay(current_batch,total_batch,warmup_ratio,lr):

    warmup_batch = total_batch * warmup_ratio

    if current_batch< warmup_batch:
        # y=kx
        k = lr/warmup_batch
        x = current_batch
        return  k * x
    else:
        # progress: 表示衰减过程，从0到1
        progress = (current_batch - warmup_batch)  / ( total_batch -warmup_batch)
        # progress从0到1的过程，cos从最大值，降到最小值
        # cos(π * progress)，从1到-1 
        # （cos(π * progress)+1）*0.5 ，从1到0，表示衰减的程度
        decay_level = (np.cos(np.pi * progress) + 1) * 0.5

        return lr * decay_level



## 4.2 验证方法

In [ ]:
def eval_model(model,sft_config:SFTConfig):

    device = next(model.parameters()).device

    model.eval()

    eval_data = get_eval_data(sft_config)

    total_batch = (len(eval_data) + sft_config.batch_size - 1)  // sft_config.batch_size
    all_batch_loss = []
    for current_batch in  range(total_batch):

        current_batch_data = eval_data[current_batch*sft_config.batch_size : (current_batch+1) * sft_config.batch_size]

        max_length = max([len(sample) for sample in current_batch_data])

        for sample in current_batch_data:
            padding_length = max_length - len(sample)
            sample.extend([tokenizer.pad_token_id] * padding_length)

        data_tensor = torch.tensor(current_batch_data, dtype=torch.long).to(device)
        # input_ids:
        input_ids = data_tensor[:,:-1]
        labels = data_tensor[:,1:]
        assistant_mask = create_answer_mask(labels=labels,tokenizer=tokenizer)
        with torch.no_grad():
            logits = model(input_ids).logits

        batch_loss = compute_loss(logits=logits, labels=labels,assistant_mask=assistant_mask )

        all_batch_loss.append(batch_loss.item())

    average_loss = sum(all_batch_loss) / len(all_batch_loss)

    return average_loss



In [ ]:
from device_utils import get_device

from torch.utils.tensorboard.writer import SummaryWriter
import tqdm


def train(sft_config:SFTConfig):
    """
    训练主流程：
    4.2.0 初始化模型，优化器，获取总的训练数据 ✅
    4.2.1 构造模型前向传播的输入，input_ids，labels，对input_ids做padding，构造assistant_answer_mask ✅
    4.2.2 前向传播，获取到logits ✅
    4.2.3 基于logits,assistant_answer_mask,labels，算损失 ✅
    4.2.4 反向传播：计算梯度 ✅
    4.2.5 做一个学习率调度 ✅
    4.2.6 基于新的学习率，做参数更新 ✅
    """
    # 初始化模型
    from transformers import AutoModelForCausalLM
    from torch.optim.adamw import AdamW # 对于大模型微调，一般使用AdamW
    device = get_device()
    print(f"using device: {device}")
    model = AutoModelForCausalLM.from_pretrained(org_model_name)
    model.to(device)
    model.train()
    optimizer = AdamW(model.parameters(), lr=sft_config.lr)
    loss_list = []


    # todo: 构建一个获取数据的方法
    # data: 第一个list，是所有数据，第二个list，是一条数据的message_list
    data: list[list] = get_train_data(sft_config)
    total_batch = (len(data) + sft_config.batch_size - 1)  // sft_config.batch_size

    # 构建tensorsorboard的writer对象，和tqdm对象
    writer = SummaryWriter(log_dir=sft_config.log_dir)
    progress_bar = tqdm.tqdm(total=total_batch)
    for current_batch in range(total_batch):
        # data[0:4], data[4:8], data[8:12] ... 
        current_batch_data = data[current_batch*sft_config.batch_size : (current_batch+1) * sft_config.batch_size]

        max_length = max([len(sample) for sample in current_batch_data])

        for sample in current_batch_data:
            padding_length = max_length - len(sample)
            sample.extend([tokenizer.pad_token_id] * padding_length)
        
        data_tensor = torch.tensor(current_batch_data, dtype=torch.long).to(device)
        # input_ids:
        input_ids = data_tensor[:,:-1]
        labels = data_tensor[:,1:]

        # 找到labels当中，哪些token_ids是pad_token_id

        # padding_mask的含义：为1的地方，不是pad_token, 为0的地方，是pad_token
        padding_mask = torch.where(labels == tokenizer.pad_token_id, 0, 1)

        assistant_mask = create_answer_mask(labels=labels,tokenizer=tokenizer)

        # 将padding_mask和assistant_mask取一个交集，交集之后，为1的地方，才是最终，需要计算损失的地方，交集之后为0的，不需要算损失
        final_mask = assistant_mask & padding_mask


        logits = model(input_ids).logits #model前向传播，此处得到的结果是一个对象，需要通过.logits方式，获取到logits属性

        loss = compute_loss(logits=logits,labels=labels,assistant_mask=final_mask)
        loss_list.append(loss.item())
        loss.backward()

        current_lr = cosine_decay(current_batch,total_batch,sft_config.warmup_ratio,sft_config.lr)

        optimizer.param_groups[0]["lr"] = current_lr

        optimizer.step()

        optimizer.zero_grad()


        should_eval = current_batch % sft_config.eval_iter == 0
        should_log = current_batch % sft_config.log_iter  == 0
        if should_eval:
            average_loss = eval_model(model,sft_config)
            writer.add_scalar("eval/loss",average_loss,current_batch)
            model.train()

        if should_log:
            last_iter_loss = loss_list[-sft_config.log_iter:]
            average_loss = sum(last_iter_loss) / len(last_iter_loss)

            writer.add_scalar("train/loss",average_loss,current_batch)

            writer.add_scalar("train/current_lr",current_lr, current_batch)

        progress_bar.update(1)
        progress_bar.set_postfix(loss = f"{loss.item():.5f}", current_lr = f"{current_lr:.2e}")



    model.save_pretrained(sft_config.save_dir)
    tokenizer.save_pretrained(sft_config.save_dir)



